# 01 — Training Walkthrough

Train the DenseNet121 classifier on NIH ChestX-ray14.

**Covers**: data loading, model init, training loop, checkpoint saving.

> **T4 GPU (Kaggle/Colab)**: set `TRAINING_MODE=kaggle` or `TRAINING_MODE=full`
> **Laptop (CPU)**: set `TRAINING_MODE=subset` for a fast smoke-test

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Configure Training Mode

Change `TRAINING_MODE` to match your environment.

In [ ]:
import os
os.environ['TRAINING_MODE'] = 'subset'   # 'subset' | 'full' | 'kaggle'
os.environ['SUBSET_SIZE']   = '500'
os.environ['TRAIN_EPOCHS']  = '3'

from src.utils.config import TrainingConfig
print(f'Mode        : {TrainingConfig.mode}')
print(f'Subset size : {TrainingConfig.subset_size}')
print(f'Epochs      : {TrainingConfig.epochs}')
print(f'Batch size  : {TrainingConfig.batch_size()}')
print(f'Device      : {"cuda" if torch.cuda.is_available() else "cpu"}')

## 3. Load Data

In [ ]:
from src.data.extract import prepare_data
from src.data.dataloader import build_dataloaders

labels_df, image_dir = prepare_data()
train_loader, val_loader, test_loader = build_dataloaders(
    labels_df, image_dir,
    subset_size=TrainingConfig.subset_size,
)
print(f'Train batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')
print(f'Test batches  : {len(test_loader)}')

## 4. Run Training

In [ ]:
from src.training.train import train

results = train(
    resume  = False,
    dry_run = False,   # Set True for a 1-batch smoke test
)
print(f"Best val AUC: {results['best_auc']:.4f}")

## 5. Kaggle Full Training

To train on the full NIH dataset using Kaggle's free T4 GPU:

1. Upload this notebook to Kaggle
2. Add dataset: `nih-chest-xrays` (pre-mounted)
3. Set `TRAINING_MODE=kaggle` and `TRAIN_EPOCHS=10`
4. Run all cells
5. Download `best_model.pt` from the output tab

In [ ]:
# Kaggle-specific setup
import os
if os.path.exists('/kaggle/input'):
    os.environ['TRAINING_MODE'] = 'kaggle'
    os.environ['TRAIN_EPOCHS']  = '10'
    print('Running on Kaggle T4 GPU')
else:
    print('Not on Kaggle — using subset mode')